In [14]:
import pandas as pd

data4=pd.read_csv("data/Baron/GSM2230760_human4_umifm_counts.csv",header=0,index_col=0)
data1=pd.read_csv("data/Baron/GSM2230757_human1_umifm_counts.csv",header=0,index_col=0)
data2=pd.read_csv("data/Baron/GSM2230758_human2_umifm_counts.csv",header=0,index_col=0)
data3=pd.read_csv("data/Baron/GSM2230759_human3_umifm_counts.csv",header=0,index_col=0)

for data in [data1,data2,data3,data4]:
    print(data.shape)

(1937, 20127)
(1724, 20127)
(3605, 20127)
(1303, 20127)


In [12]:
for data in [data1,data2,data3,data4]:
    print(data.head())

                    Unnamed: 0              barcode assigned_cluster  A1BG  \
0  human1_lib1.final_cell_0001  GATGACGGAC-GGTGGGAT           acinar     0   
1  human1_lib1.final_cell_0002  GAGCGTTGCT-ACCTTCTT           acinar     0   
2  human1_lib1.final_cell_0003    CTTACGGG-CCATTACT           acinar     0   
3  human1_lib1.final_cell_0004  GATGTACACG-TTAAACTG           acinar     0   
4  human1_lib1.final_cell_0005  GAGATTGCGA-GTCGTCGT           acinar     0   

   A1CF  A2M  A2ML1  A4GALT  A4GNT  AA06  ...  ZWILCH  ZWINT  ZXDA  ZXDB  \
0     4    0      0       0      0     0  ...       0      0     0     0   
1     0    0      0       0      0     0  ...       0      0     0     0   
2     0    0      0       0      0     0  ...       0      0     0     0   
3     0    0      0       0      0     0  ...       1      0     0     0   
4     0    0      0       0      0     0  ...       0      0     0     0   

   ZXDC  ZYG11B  ZYX  ZZEF1  ZZZ3  pk  
0     0       0    2      0     0 

In [22]:
import h5py
import anndata as ad
path="data/Quake_Smart-seq2_Lung/Quake_Smart-seq2_Lung.h5"
with h5py.File(path, "r") as f:
        X = f["X"][:]
        y = f["Y"][:]

    # build AnnData
adata = ad.AnnData(X)
adata.obs["label"] = y

In [23]:
adata

AnnData object with n_obs × n_vars = 1676 × 23341
    obs: 'label'

In [ ]:
"""
Gaussian Mixture Copula Model (GMCM) for cell clustering
from VGAE latent representations of single-cell RNA-seq data.

Architecture:
    GCNEncoder  →  z (latent)  →  GMCM  →  cluster assignments

Copula idea: transform z to uniform marginals (CDF), then model the
joint dependence structure with a Gaussian mixture in copula space.
This decouples marginal distributions from their dependency structure,
giving richer cluster shapes than a plain GMM.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops
from torch_geometric.typing import Adj
from torch_sparse import SparseTensor

# ──────────────────────────────────────────────────────────────────────────────
# 1.  GCN building block
# ──────────────────────────────────────────────────────────────────────────────

def gnc_encoder(in_ch: int, out_ch: int) -> GCNConv:
    return GCNConv(in_ch, out_ch, cached=True, add_self_loops=True)


# ──────────────────────────────────────────────────────────────────────────────
# 2.  Three-layer variational GCN encoder
# ──────────────────────────────────────────────────────────────────────────────

class GCNEncoder(nn.Module):
    """
    Variational GCN encoder: maps (x, edge_index) → (mu, log_var).

    Args:
        in_channels:     Input node-feature dimensionality.
        hidden_channels: Hidden layer width.
        latent_channels: Latent space dimensionality.
        activation:      Post-conv non-linearity.
        dropout:         Dropout probability (0 = off).
    """

    def __init__(
        self,
        in_channels:     int,
        hidden_channels: int,
        latent_channels: int,
        activation:      nn.Module = nn.ReLU(),
        dropout:         float     = 0.0,
    ) -> None:
        super().__init__()
        self.activation = activation
        self.dropout    = nn.Dropout(p=dropout)

        self.conv1      = gnc_encoder(in_channels,     hidden_channels)
        self.conv2      = gnc_encoder(hidden_channels, hidden_channels)
        self.conv_mu    = gnc_encoder(hidden_channels, latent_channels)
        self.conv_logv  = gnc_encoder(hidden_channels, latent_channels)

    # ------------------------------------------------------------------
    def _encode(self, x: Tensor, edge_index: Adj) -> Tensor:
        h = self.dropout(self.activation(self.conv1(x, edge_index)))
        h = self.dropout(self.activation(self.conv2(h, edge_index)))
        return h

    def forward(self, x: Tensor, edge_index: Adj) -> tuple[Tensor, Tensor]:
        h       = self._encode(x, edge_index)
        mu      = self.conv_mu(h, edge_index)
        log_var = self.conv_logv(h, edge_index)
        return mu, log_var

    # ------------------------------------------------------------------
    @staticmethod
    def reparameterise(mu: Tensor, log_var: Tensor) -> Tensor:
        std = (0.5 * log_var).exp()
        return mu + std * torch.randn_like(std)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  Gaussian Mixture Copula Model  (GMCM)
# ──────────────────────────────────────────────────────────────────────────────

class GaussianMixtureCopula(nn.Module):
    """
    Soft Gaussian Mixture Model operating in *copula space*.

    Steps
    -----
    1. Normalise z → u ∈ (0,1)^D  via the empirical CDF (rank transform).
    2. Map u back to Gaussian quantiles:  v = Φ⁻¹(u)  (normal scores).
    3. Fit a full-covariance GMM on v with learnable (π, μ, Σ).

    The log-likelihood is:
        log p(v) = log Σ_k  π_k · N(v | μ_k, Σ_k)  +  log|J(u→v)|

    The Jacobian term is the sum of standard-normal log-PDFs evaluated at v,
    which is constant w.r.t. cluster parameters and can be dropped for EM /
    gradient-based optimisation of the mixture part.

    Args:
        latent_dim:  Dimensionality of z (= D).
        n_clusters:  Number of mixture components (K).
        reg_covar:   Regularisation added to diagonal of Σ for stability.
    """

    def __init__(
        self,
        latent_dim: int,
        n_clusters: int,
        reg_covar:  float = 1e-4,
    ) -> None:
        super().__init__()
        self.K         = n_clusters
        self.D         = latent_dim
        self.reg_covar = reg_covar

        # Mixture weights (un-normalised logits)
        self.log_pi = nn.Parameter(torch.zeros(n_clusters))

        # Per-component means
        self.mu = nn.Parameter(torch.randn(n_clusters, latent_dim) * 0.1)

        # Per-component Cholesky factors  L  s.t.  Σ = L Lᵀ + reg·I
        # Stored as lower-triangular matrix (flattened)
        self.L_raw = nn.Parameter(
            torch.eye(latent_dim).unsqueeze(0).repeat(n_clusters, 1, 1)
        )

    # ------------------------------------------------------------------
    def _cholesky(self) -> Tensor:
        """Returns lower-triangular Cholesky L with positive diagonal."""
        L = torch.tril(self.L_raw)
        diag_idx = torch.arange(self.D, device=L.device)
        L[:, diag_idx, diag_idx] = F.softplus(L[:, diag_idx, diag_idx]) + 1e-6
        return L  # (K, D, D)

    # ------------------------------------------------------------------
    @staticmethod
    def _rank_transform(z: Tensor) -> Tensor:
        """
        Empirical CDF rank transform:  z → u ∈ (ε, 1-ε)^D
        Applied dimension-wise.  Shape: (N, D) → (N, D).
        """
        N = z.size(0)
        # argsort twice gives rank (0-based)
        ranks = z.argsort(0).argsort(0).float()          # (N, D)
        u = (ranks + 1) / (N + 1)                        # avoid 0/1
        return u

    # ------------------------------------------------------------------
    @staticmethod
    def _normal_scores(u: Tensor) -> Tensor:
        """Φ⁻¹(u)  — quantile function of N(0,1)."""
        return torch.erfinv(2 * u - 1) * (2 ** 0.5)

    # ------------------------------------------------------------------
    def _log_component_density(self, v: Tensor) -> Tensor:
        """
        log N(v | μ_k, Σ_k)  for each component k.

        Returns
        -------
        Tensor of shape (N, K).
        """
        L    = self._cholesky()                           # (K, D, D)
        diff = v.unsqueeze(1) - self.mu.unsqueeze(0)     # (N, K, D)

        # Solve L @ alpha = diff  →  alpha = L⁻¹ diff
        # torch.linalg.solve_triangular expects (..., D, D), (..., D, N)
        diff_T  = diff.permute(1, 2, 0)                  # (K, D, N)
        alpha   = torch.linalg.solve_triangular(L, diff_T, upper=False)
        maha    = (alpha ** 2).sum(1).T                   # (N, K)

        log_det = L.diagonal(dim1=-2, dim2=-1).log().sum(-1)  # (K,)
        log_2pi = self.D * 0.9189385332  # D/2 * log(2π)

        return -0.5 * maha - log_det - log_2pi            # (N, K)

    # ------------------------------------------------------------------
    def log_likelihood(self, z: Tensor) -> Tensor:
        """
        GMCM log-likelihood (copula Jacobian dropped).

        Returns scalar.
        """
        u = self._rank_transform(z)
        v = self._normal_scores(u)

        log_pi = F.log_softmax(self.log_pi, dim=0)        # (K,)
        log_p  = self._log_component_density(v)           # (N, K)
        return torch.logsumexp(log_pi + log_p, dim=1).mean()

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict(self, z: Tensor) -> Tensor:
        """Hard cluster assignment.  Returns (N,) integer tensor."""
        u = self._rank_transform(z)
        v = self._normal_scores(u)
        log_pi = F.log_softmax(self.log_pi, dim=0)
        log_p  = self._log_component_density(v)
        return (log_pi + log_p).argmax(dim=1)

    # ------------------------------------------------------------------
    def soft_assign(self, z: Tensor) -> Tensor:
        """Posterior responsibilities q(k | z_i).  Returns (N, K)."""
        u = self._rank_transform(z)
        v = self._normal_scores(u)
        log_pi = F.log_softmax(self.log_pi, dim=0)
        log_p  = self._log_component_density(v)
        return F.softmax(log_pi + log_p, dim=1)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  Full VGAE + GMCM model
# ──────────────────────────────────────────────────────────────────────────────

class VGAEwithGMCM(nn.Module):
    """
    End-to-end model:
        GCNEncoder  →  z  →  GaussianMixtureCopula
        + inner-product decoder for edge reconstruction.

    Loss
    ----
        L = L_recon  +  β · L_KL  −  γ · L_copula

    where
        L_recon  = binary cross-entropy on reconstructed adjacency
        L_KL     = KL( q(z|x) ‖ N(0,I) )
        L_copula = GMCM log-likelihood (maximised)
    """

    def __init__(
        self,
        in_channels:     int,
        hidden_channels: int,
        latent_channels: int,
        n_clusters:      int,
        dropout:         float = 0.0,
        beta:            float = 1.0,
        gamma:           float = 1.0,
    ) -> None:
        super().__init__()
        self.encoder = GCNEncoder(in_channels, hidden_channels, latent_channels, dropout=dropout)
        self.gmcm    = GaussianMixtureCopula(latent_channels, n_clusters)
        self.beta    = beta
        self.gamma   = gamma

    # ------------------------------------------------------------------
    def forward(
        self, x: Tensor, edge_index: Adj
    ) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        mu, log_var = self.encoder(x, edge_index)
        z           = GCNEncoder.reparameterise(mu, log_var)
        z_hat       = z @ z.T                              # inner-product decoder
        return z, z_hat, mu, log_var

    # ------------------------------------------------------------------
    def loss(
        self,
        x:          Tensor,
        edge_index: Adj,
        adj_label:  Tensor,                               # (N, N) dense 0/1
        norm_weight: float  = 1.0,
        pos_weight:  Tensor | None = None,
    ) -> tuple[Tensor, dict[str, float]]:
        """
        Compute total loss and return a dict of named components.

        Args:
            x:           Node features   (N, F).
            edge_index:  COO edge index.
            adj_label:   Dense adjacency used as reconstruction target.
            norm_weight: Scaling for L_recon (accounts for graph sparsity).
            pos_weight:  BCE positive-class weight for imbalanced graphs.

        Returns:
            total_loss, {'recon': ..., 'kl': ..., 'copula': ...}
        """
        z, z_hat, mu, log_var = self(x, edge_index)

        # ── Reconstruction ──────────────────────────────────────────────
        l_recon = norm_weight * F.binary_cross_entropy_with_logits(
            z_hat, adj_label, pos_weight=pos_weight, reduction="mean"
        )

        # ── KL divergence  q(z|x) ‖ N(0,I) ────────────────────────────
        l_kl = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).mean()

        # ── Copula clustering (maximise → minimise negative) ────────────
        l_copula = -self.gmcm.log_likelihood(z)

        total = l_recon + self.beta * l_kl + self.gamma * l_copula

        return total, {
            "recon":  l_recon.item(),
            "kl":     l_kl.item(),
            "copula": l_copula.item(),
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def cluster(self, x: Tensor, edge_index: Adj) -> Tensor:
        """Infer hard cluster labels for all cells."""
        self.eval()
        mu, _ = self.encoder(x, edge_index)
        return self.gmcm.predict(mu)


# ──────────────────────────────────────────────────────────────────────────────
# 5.  Use-case: synthetic scRNA-seq data
# ──────────────────────────────────────────────────────────────────────────────

def build_knn_graph(x: Tensor, k: int = 10) -> Tensor:
    """Build a simple cosine-similarity KNN graph from cell-by-gene matrix."""
    x_norm  = F.normalize(x, dim=1)
    sim     = x_norm @ x_norm.T
    topk    = sim.topk(k + 1, dim=1).indices[:, 1:]          # exclude self

    src = torch.arange(x.size(0), device=x.device).repeat_interleave(k)
    dst = topk.reshape(-1)
    return torch.stack([src, dst], dim=0)                     # (2, N*k)


def demo() -> None:
    torch.manual_seed(42)

    # ── Synthetic data ───────────────────────────────────────────────────────
    N_CELLS, N_GENES = 500, 200
    N_CLUSTERS       = 5
    LATENT_DIM       = 16

    # Simulate clusters by sampling from Gaussian blobs
    labels_true = torch.randint(0, N_CLUSTERS, (N_CELLS,))
    centers     = torch.randn(N_CLUSTERS, N_GENES) * 3
    x           = centers[labels_true] + torch.randn(N_CELLS, N_GENES) * 0.5
    x           = F.relu(x)                                   # non-negative counts

    edge_index  = build_knn_graph(x, k=15)

    # Dense adjacency for reconstruction loss (small demo → feasible)
    adj_label = torch.zeros(N_CELLS, N_CELLS)
    adj_label[edge_index[0], edge_index[1]] = 1.0

    # Normalisation factor: ratio of all edges to positives (standard VGAE trick)
    n_pos       = adj_label.sum()
    norm_weight = N_CELLS ** 2 / (2 * (N_CELLS ** 2 - n_pos))
    pos_weight  = torch.tensor((N_CELLS ** 2 - n_pos) / n_pos)

    # ── Model & optimiser ────────────────────────────────────────────────────
    model = VGAEwithGMCM(
        in_channels     = N_GENES,
        hidden_channels = 64,
        latent_channels = LATENT_DIM,
        n_clusters      = N_CLUSTERS,
        dropout         = 0.1,
        beta            = 1.0,
        gamma           = 0.5,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # ── Training loop ────────────────────────────────────────────────────────
    model.train()
    for epoch in range(1, 101):
        optimizer.zero_grad()
        loss, parts = model.loss(x, edge_index, adj_label, norm_weight, pos_weight)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(
                f"Epoch {epoch:>3} | "
                f"loss={loss.item():.4f}  "
                f"recon={parts['recon']:.4f}  "
                f"kl={parts['kl']:.4f}  "
                f"copula={parts['copula']:.4f}"
            )

    # ── Evaluation ───────────────────────────────────────────────────────────
    labels_pred = model.cluster(x, edge_index)
    print(f"\nPredicted cluster distribution: {labels_pred.bincount().tolist()}")

    # Soft assignments (responsibilities)
    model.eval()
    with torch.no_grad():
        mu, _ = model.encoder(x, edge_index)
        resp  = model.gmcm.soft_assign(mu)          # (N, K)
    print(f"Mean cluster entropy: {(-resp * resp.clamp(1e-8).log()).sum(1).mean():.4f}")


if __name__ == "__main__":
    demo()

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


class DifferentiableGMCM(nn.Module):
    """
    Differentiable approximation of a Gaussian Mixture Copula Model.

    Main idea:
    - Replace hard empirical rank transform with a smooth empirical CDF
      based on pairwise sigmoids.
    - Transform smooth CDF values to Gaussian normal scores.
    - Fit a Gaussian mixture in that transformed space.

    This allows gradients to flow back into z.
    """

    def __init__(
        self,
        latent_dim: int,
        n_clusters: int,
        reg_covar: float = 1e-4,
        tau: float = 0.1,
        eps: float = 1e-6,
    ) -> None:
        super().__init__()
        self.K = n_clusters
        self.D = latent_dim
        self.reg_covar = reg_covar
        self.tau = tau
        self.eps = eps

        self.log_pi = nn.Parameter(torch.zeros(n_clusters))
        self.mu = nn.Parameter(torch.randn(n_clusters, latent_dim) * 0.1)

        # raw lower-triangular factors for covariance
        self.L_raw = nn.Parameter(
            torch.eye(latent_dim).unsqueeze(0).repeat(n_clusters, 1, 1)
        )

    def _cholesky(self) -> Tensor:
        """
        Build valid lower-triangular Cholesky factors.
        """
        L = torch.tril(self.L_raw)
        idx = torch.arange(self.D, device=L.device)
        L[:, idx, idx] = F.softplus(L[:, idx, idx]) + self.reg_covar
        return L

    def _smooth_empirical_cdf(self, z: Tensor) -> Tensor:
        """
        Differentiable approximation of the empirical CDF.

        For each dimension independently:
            F_hat(z_i) = mean_j sigmoid((z_i - z_j) / tau)

        z: (N, D)
        returns u: (N, D) in (0,1)
        """
        # pairwise differences per dimension
        # diff[n, m, d] = z[n, d] - z[m, d]
        diff = z.unsqueeze(1) - z.unsqueeze(0)   # (N, N, D)

        # smooth indicator I[z_j <= z_i]
        cdf_vals = torch.sigmoid(diff / self.tau).mean(dim=1)  # (N, D)

        # avoid exact 0 or 1 before inverse Gaussian CDF
        u = cdf_vals.clamp(self.eps, 1.0 - self.eps)
        return u

    def _gaussian_ppf(self, u: Tensor) -> Tensor:
        """
        Approximate inverse standard normal CDF:
            Phi^{-1}(u) = sqrt(2) * erfinv(2u - 1)
        """
        return math.sqrt(2.0) * torch.erfinv(2.0 * u - 1.0)

    def _to_normal_scores(self, z: Tensor) -> Tensor:
        """
        Differentiable copula transform.
        """
        u = self._smooth_empirical_cdf(z)
        v = self._gaussian_ppf(u)
        return v

    def _log_component_density(self, v: Tensor) -> Tensor:
        """
        Log-density of each Gaussian component.

        v: (N, D)
        returns: (N, K)
        """
        L = self._cholesky()                                # (K, D, D)
        diff = v.unsqueeze(1) - self.mu.unsqueeze(0)       # (N, K, D)

        # reshape for triangular solve
        # solve L_k * a = diff_{n,k}
        rhs = diff.permute(1, 2, 0)                        # (K, D, N)
        alpha = torch.linalg.solve_triangular(
            L, rhs, upper=False
        )                                                  # (K, D, N)

        maha = (alpha ** 2).sum(dim=1).T                   # (N, K)
        log_det = torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(dim=-1)  # (K,)

        return -0.5 * maha - log_det - 0.5 * self.D * math.log(2.0 * math.pi)

    def logits(self, z: Tensor) -> Tensor:
        """
        Unnormalized log posterior scores: (N, K)
        """
        v = self._to_normal_scores(z)
        log_pi = F.log_softmax(self.log_pi, dim=0)
        return log_pi + self._log_component_density(v)

    def loss(self, z: Tensor) -> Tensor:
        """
        Unsupervised negative log-likelihood.
        """
        logits = self.logits(z)
        return -torch.logsumexp(logits, dim=1).mean()

    def soft_assign(self, z: Tensor) -> Tensor:
        """
        Posterior responsibilities: (N, K)
        """
        return F.softmax(self.logits(z), dim=1)

    @torch.no_grad()
    def predict(self, z: Tensor) -> Tensor:
        """
        Hard cluster assignments: (N,)
        """
        return self.soft_assign(z).argmax(dim=1)